# Qwen3-14B + LoRA + AWQ + vLLM Pipeline

Steps covered in this Colab-oriented notebook:
1. Clone the repository and install dependencies.
2. Configure paths and constants.
3. Merge the LoRA adapter on CPU and quantize with AWQ.
4. Load vLLM backend helpers.
5. Run vLLM inference with the quantized model.
6. Collect batch metrics.
7. Enter an optional chat loop.
8. Clean up resources.


In [ ]:
# === 1. Environment setup (Google Colab) ===
import os
import sys
from typing import List

try:
    from IPython import get_ipython
except ImportError:  # pragma: no cover
    get_ipython = None  # type: ignore


def _require_ipython():
    ip = get_ipython() if callable(get_ipython) else None
    if ip is None:
        raise RuntimeError("IPython environment is required. Please run on Google Colab.")
    return ip


def clone_repo(url: str, target: str, branch: str | None = None) -> None:
    if os.path.exists(target):
        print("Reusing existing repository:", target)
        return
    ip = _require_ipython()
    if branch:
        print(f"Cloning repository: {url} (branch={branch})")
        ip.system(f"git clone --branch {branch} --single-branch {url} {target}")
    else:
        print("Cloning repository:", url)
        ip.system(f"git clone {url} {target}")


def pip_install(packages: List[str]) -> None:
    packages = list(packages)
    if not packages:
        return
    ip = _require_ipython()
    quoted = " ".join(f'"{pkg}"' for pkg in packages)
    print("pip install:", packages)
    ip.run_line_magic("pip", f"install --upgrade {quoted}")


REPO_URL = "https://github.com/fouga1221/llm-lab2.git"
DEFAULT_REPO_DIR = "/content/llm-lab2" if "google.colab" in sys.modules else os.path.abspath("..")
REPO_DIR = DEFAULT_REPO_DIR  # Change here if you want a different clone path.
REPO_BRANCH = "main2"  # Set to None to use the remote default branch.

BASE_PACKAGES = [
    "transformers>=4.56.0,<4.57.0",
    "peft>=0.17.0,<0.18.0",
    "vllm",
    "pyyaml",
    "autoawq",
    "accelerate",
    "safetensors",
    "datasets",
]
ADDITIONAL_PACKAGES: List[str] = []  # Append extra packages here if needed.

clone_repo(REPO_URL, REPO_DIR, branch=REPO_BRANCH)
pip_install(BASE_PACKAGES + ADDITIONAL_PACKAGES)


In [ ]:
# === 2. Path and model configuration ===
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# drive上の実体フォルダにシンボリックリンクを貼る
!ln -s /content/drive/MyDrive/ProjectForte/llm-lab-save/ /content/llm-lab-save

IS_COLAB = "google.colab" in sys.modules
REPO_ROOT = Path(REPO_DIR).resolve()
DATA_ROOT = Path("/content/llm-lab-save") if IS_COLAB else Path.cwd() / "save"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "BASE_MODEL_NAME": "Qwen/Qwen3-14B",
    "LORA_DIR": DATA_ROOT / "models" / "adapters" / "qwen3-14b-lora-ojousama",
    "MERGED_OUTPUT_DIR": DATA_ROOT / "artifacts" / "merged_qwen3_14b_ojousama",
    "AWQ_OUTPUT_DIR": DATA_ROOT / "artifacts" / "awq_qwen3_14b_ojousama",
    "EXTERNAL_MERGED_DIR": None,
    "EXTERNAL_AWQ_DIR": None,
    "PERFORM_LORA_MERGE": True,
    "PERFORM_AWQ_QUANT": True,
    "AWQ_CONFIG": {
        "w_bit": 4,
        "q_group_size": 128,
        "zero_point": True,
        "version": "GEMM",
    },
    "AWQ_CALIBRATION_SAMPLES": [
        "You are a helpful assistant. Please follow the instructions strictly.",
        "Summarise the following enterprise deck in roughly 200 Japanese characters.",
    ],
}

GENERATION = {
    "MAX_NEW_TOKENS": 256,
    "TEMPERATURE": 0.7,
    "TOP_P": 0.9,
    "REPETITION_PENALTY": 1.05,
    "STOP": ["\nUser:"],
}

CHAT = {
    "SYSTEM_PROMPT": "You are a Qwen3-14B based assistant.",
    "STOP_PHRASES": ["/exit", ":q"],
    "EXIT_COMMAND": "/exit",
}

VLLM_OPTIONS = {
    "tensor_parallel_size": 1,
    "dtype": "auto",
    "max_model_len": 4096,
    "gpu_memory_utilization": 0.90,
    "download_dir": DATA_ROOT / "model_cache",
}

CONFIG["LORA_DIR"].mkdir(parents=True, exist_ok=True)
CONFIG["MERGED_OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)
CONFIG["AWQ_OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)
VLLM_OPTIONS["download_dir"].mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"BASE_MODEL_NAME: {CONFIG['BASE_MODEL_NAME']}")
print(f"LORA_DIR: {CONFIG['LORA_DIR']}")
print("To reuse existing artefacts, set the PERFORM_* flags to False and point EXTERNAL_* to their locations if needed.")


In [ ]:
# === 3. LoRA merge on CPU and AWQ quantisation ===

import gc
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from awq import AutoAWQForCausalLM
from src.llmlab.utils.awq import prepare_awq_calib_data

base_model_name = CONFIG["BASE_MODEL_NAME"]
lora_dir = CONFIG["LORA_DIR"]
merged_dir = CONFIG["MERGED_OUTPUT_DIR"]
awq_dir = CONFIG["AWQ_OUTPUT_DIR"]
external_merged_dir = CONFIG.get("EXTERNAL_MERGED_DIR")
external_awq_dir = CONFIG.get("EXTERNAL_AWQ_DIR")

if external_merged_dir:
    external_merged_dir = Path(external_merged_dir)
if external_awq_dir:
    external_awq_dir = Path(external_awq_dir)

source_model_for_awq = base_model_name

if CONFIG["PERFORM_LORA_MERGE"]:
    if not lora_dir.exists():
        raise FileNotFoundError(f"LoRA directory not found: {lora_dir}")
    print("Merging LoRA into the base model on CPU...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        device_map={"": "cpu"},
        torch_dtype=torch.float32,
        trust_remote_code=True,
        low_cpu_mem_usage=False,
    )
    lora_model = PeftModel.from_pretrained(base_model, str(lora_dir))
    merged_model = lora_model.merge_and_unload()
    merged_model.save_pretrained(merged_dir, safe_tensors=True)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    tokenizer.save_pretrained(merged_dir)
    del base_model, lora_model, merged_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    source_model_for_awq = str(merged_dir)
else:
    candidate_merges = [external_merged_dir, merged_dir]
    for candidate in candidate_merges:
        if candidate and candidate.exists() and any(candidate.iterdir()):
            print("Using prebuilt merged weights:", candidate)
            source_model_for_awq = str(candidate)
            break
    else:
        print("No merged weights found; falling back to the base model.")

if CONFIG["PERFORM_AWQ_QUANT"]:
    target_awq_dir = external_awq_dir or awq_dir
    target_awq_dir.mkdir(parents=True, exist_ok=True)
    if any(target_awq_dir.glob("*.safetensors")):
        print("Using existing AWQ artefacts:", target_awq_dir)
    else:
        print("Quantising with AWQ (this may take some time)...")
        tokenizer = AutoTokenizer.from_pretrained(source_model_for_awq, trust_remote_code=True)
        awq_model = AutoAWQForCausalLM.from_pretrained(
            source_model_for_awq,
            trust_remote_code=True,
            device_map={"": "cpu"},
        )
        calib_data_raw = CONFIG["AWQ_CALIBRATION_SAMPLES"]
        calib_data, calib_text_column = prepare_awq_calib_data(calib_data_raw)
        awq_model.quantize(
            tokenizer,
            quant_config=CONFIG["AWQ_CONFIG"],
            calib_data=calib_data,
            text_column=calib_text_column,
        )
        awq_model.save_quantized(target_awq_dir, use_safetensors=True)
        tokenizer.save_pretrained(target_awq_dir)
        del awq_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    VLLM_MODEL_PATH = str(target_awq_dir)
    VLLM_QUANTIZATION = "awq"
else:
    candidate_awq_dirs = [external_awq_dir, awq_dir]
    selected_awq = None
    for candidate in candidate_awq_dirs:
        if candidate and candidate.exists() and any(candidate.glob("*.safetensors")):
            selected_awq = candidate
            break
    if selected_awq:
        print("Using prebuilt AWQ artefacts:", selected_awq)
        VLLM_MODEL_PATH = str(selected_awq)
        VLLM_QUANTIZATION = "awq"
    else:
        print("AWQ artefacts not found; using the non-quantised source model.")
        VLLM_MODEL_PATH = source_model_for_awq
        VLLM_QUANTIZATION = "none"

print("Model path for vLLM:", VLLM_MODEL_PATH)
print("vLLM quantisation mode:", VLLM_QUANTIZATION)



In [ ]:
# === 4. Load vLLM backend helpers ===
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.llmlab.backends.vllm_backend import (
    ModelBundle,
    chat_loop,
    free_model,
    load_model,
    profile_generation,
)

print("vLLM backend utilities loaded.")


In [ ]:
# === 5. Load quantised model into vLLM ===
cfg = {
    "model_name": VLLM_MODEL_PATH,
    "tensor_parallel_size": VLLM_OPTIONS["tensor_parallel_size"],
    "dtype": VLLM_OPTIONS["dtype"],
    "max_model_len": VLLM_OPTIONS["max_model_len"],
    "gpu_memory_utilization": VLLM_OPTIONS["gpu_memory_utilization"],
    "download_dir": str(VLLM_OPTIONS["download_dir"]),
    "quantization": VLLM_QUANTIZATION,
    "lora_path": None,
    "merge_lora": False,
}

print("vLLM load config:", cfg)
bundle: ModelBundle = load_model(cfg)
print("Model loaded. Tokenizer available:", bool(bundle["tok"]))
print("Load timings:", bundle["cfg"].get("_timings", {}))


In [ ]:
# === 6. Batch inference and metrics ===
import pandas as pd

prompts = [
    "Summarise the following specification in Japanese:\n- Multi-stage LoRA tuned Qwen3-14B\n- Target deployment on lightweight edge devices",
    "Explain three benefits of deploying this LoRA-enhanced model as a customer-facing FAQ bot.",
]

outputs, metrics = profile_generation(
    bundle,
    prompts,
    max_new_tokens=GENERATION["MAX_NEW_TOKENS"],
    temperature=GENERATION["TEMPERATURE"],
    top_p=GENERATION["TOP_P"],
    repetition_penalty=GENERATION["REPETITION_PENALTY"],
    stop=GENERATION["STOP"],
)

display(pd.DataFrame({"prompt": prompts, "output": outputs}))
display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))


In [ ]:
# === 7. Interactive chat (optional) ===
print(f"Chat session started. Type {CHAT['EXIT_COMMAND']} to exit.")
try:
    chat_loop(
        bundle,
        system_prompt=CHAT["SYSTEM_PROMPT"],
        stop_phrases=list({*CHAT["STOP_PHRASES"], CHAT["EXIT_COMMAND"]}),
        max_new_tokens=GENERATION["MAX_NEW_TOKENS"],
        temperature=GENERATION["TEMPERATURE"],
        top_p=GENERATION["TOP_P"],
        repetition_penalty=GENERATION["REPETITION_PENALTY"],
    )
finally:
    print("Chat session finished.")


In [ ]:
# === 8. Cleanup ===
free_model(bundle)
print("Freed vLLM model and tokenizer.")
